# TLeakBench — LLM evaluation
**"Can Generative AI Catch Temporal Data Leakage?"**

**Setup (once):**
**Kaggle:** upload the files as a Dataset, set Accelerator = GPU T4×2 and Internet = On, then *Save Version → Save & Run All*.

**Colab:**
1. Create `MyDrive/TLeakBench/` in Google Drive and put `tleakbench_generator.py` into it.
2. (Optional) For real data, put a discharge CSV and an ERA5-Land (T, P) CSV into `MyDrive/TLeakBench/data/`.
3. Runtime → Change runtime type → **GPU** (T4 is enough; A100/L4 are faster).
4. Adjust the settings in cell 1 and run the cells in order.

Results are written after every batch, so an interrupted run resumes where it stopped.

In [ ]:
# ===================== 1. SETTINGS =====================
import os, sys, json, re, time, random, gc, signal, ast, textwrap

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if IN_COLAB:
    DRIVE_DIR = "/content/drive/MyDrive/TLeakBench"
elif IN_KAGGLE:
    DRIVE_DIR = "/kaggle/working/TLeakBench"     # writable folder on Kaggle
else:
    DRIVE_DIR = "./TLeakBench"
BENCH_DIR = f"{DRIVE_DIR}/bench"        # benchmark files
RES_DIR   = f"{DRIVE_DIR}/results"      # LLM outputs and metrics
DATA_DIR  = f"{DRIVE_DIR}/data"         # real data
for d in (DRIVE_DIR, BENCH_DIR, RES_DIR, DATA_DIR):
    os.makedirs(d, exist_ok=True)

if IN_KAGGLE:
    # copy generator, benchmark, previous outputs and data from the Kaggle Dataset (/kaggle/input)
    import glob, shutil
    for src_f in glob.glob("/kaggle/input/**/*", recursive=True):
        if os.path.isdir(src_f):
            continue
        fn = os.path.basename(src_f)
        if fn == "tleakbench_generator.py":
            dst = f"{DRIVE_DIR}/{fn}"
        elif fn in ("tleakbench.jsonl", "twins.jsonl") or (fn == "oracle_summary.csv" and "/bench/" in src_f):
            dst = f"{BENCH_DIR}/{fn}"
        elif fn.startswith("raw_") and fn.endswith(".jsonl"):
            dst = f"{RES_DIR}/{fn}"
        elif "/data/" in src_f and fn.endswith(".csv"):
            dst = f"{DATA_DIR}/{fn}"
        else:
            continue
        if not os.path.exists(dst):
            shutil.copy(src_f, dst)
            print("copied:", dst)

# --- Real data (None = synthetic series) ---
DISCHARGE_CSV = None            # e.g. f"{DATA_DIR}/discharge_gauge.csv"
Q_DATE_COL, Q_VALUE_COL = "date", "discharge"
Q_STATION_COL, STATION_ID = None, None   # multi-gauge file: column name and gauge id
MET_CSV = None                  # daily ERA5-Land, e.g. f"{DATA_DIR}/era5land_gauge.csv"
MET_DATE_COL, T_COL, P_COL = "date", "t2m", "tp"
REBUILD_BENCH = False           # True: regenerate the benchmark

# --- LLM experiment ---
MODELS = [
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    "Qwen/Qwen2.5-Coder-3B-Instruct",
    # "deepseek-ai/deepseek-coder-6.7b-instruct",
]
METHODS = ["B0", "B1", "B2", "B4"]   # B3 (static rules) needs no LLM
SC_K = 1                 # self-consistency: 1 = greedy; 5 = confidence estimate (5x slower)
BATCH_SIZE = 8           # 4–8 on T4, 16 on A100
MAX_NEW_TOKENS = 384
QUICK_TEST = 0           # >0: test run on this many items (e.g. 16)
DRY_RUN = False          # True: pipeline test without GPU (fake LLM)
SEED = 2026
random.seed(SEED)
print("Working folder:", DRIVE_DIR)

In [ ]:
# ===================== 2. PACKAGES =====================
!pip -q install -U "transformers>=4.44" accelerate bitsandbytes statsmodels

In [ ]:
import numpy as np
import pandas as pd

# ===================== 3. PREPARE REAL DATA (optional) =====================
def to_dekadal(df, date_col, agg):
    """Aggregate a daily/dekadal series to dekads (date = start of dekad: 1, 11, 21)."""
    d = df.copy()
    dt = pd.to_datetime(d[date_col])
    start = np.where(dt.dt.day <= 10, 1, np.where(dt.dt.day <= 20, 11, 21))
    d["date"] = pd.to_datetime(dict(year=dt.dt.year, month=dt.dt.month, day=start))
    return d.groupby("date").agg(agg).reset_index()

PREPARED_CSV = None
if DISCHARGE_CSV:
    q = pd.read_csv(DISCHARGE_CSV)
    if Q_STATION_COL:
        q = q[q[Q_STATION_COL].astype(str) == str(STATION_ID)]
    q = q[[Q_DATE_COL, Q_VALUE_COL]].rename(columns={Q_VALUE_COL: "Q"})
    q = to_dekadal(q, Q_DATE_COL, {"Q": "mean"})
    if not MET_CSV:
        raise ValueError("MET_CSV (ERA5-Land T and P) is required: the templates use columns T and P.")
    m = pd.read_csv(MET_CSV)
    m = m[[MET_DATE_COL, T_COL, P_COL]].rename(columns={T_COL: "T", P_COL: "P"})
    if m["T"].mean() > 200:
        m["T"] = m["T"] - 273.15
        print("T: Kelvin -> degC")
    if m["P"].max() < 1:
        m["P"] = m["P"] * 1000
        print("P: m -> mm")
    m = to_dekadal(m, MET_DATE_COL, {"T": "mean", "P": "sum"})
    data = q.merge(m, on="date", how="inner").sort_values("date").reset_index(drop=True)
    print(f"Prepared: {len(data)} dekads ({data.date.min().date()} — {data.date.max().date()}), "
          f"missing Q: {data.Q.isna().sum()}")
    if len(data) < 396:
        print("Warning: fewer than 11 years of data may be too short for STL (period=36) and climatology.")
    PREPARED_CSV = f"{DATA_DIR}/prepared_dekadal.csv"
    data.to_csv(PREPARED_CSV, index=False)
else:
    print("No real data given: using the synthetic series.")

In [ ]:
# ===================== 4. BENCHMARK =====================
gen_path = f"{DRIVE_DIR}/tleakbench_generator.py"
assert os.path.exists(gen_path), f"{gen_path} not found: put the generator file into the working folder."
sys.path.insert(0, DRIVE_DIR)
import importlib
import tleakbench_generator as G
importlib.reload(G)
G.CFG["out_dir"] = BENCH_DIR
G.CFG["data_csv"] = PREPARED_CSV
G.CFG["seed"] = SEED

bench_file = f"{BENCH_DIR}/tleakbench.jsonl"
if REBUILD_BENCH or not os.path.exists(bench_file):
    G.main()

def read_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

BENCH = read_jsonl(bench_file)
TWINS = {t["pair_id"]: t for t in read_jsonl(f"{BENCH_DIR}/twins.jsonl")}
DATA_DF, DATA_SRC = G.load_data()
ITEMS = BENCH[:QUICK_TEST] if QUICK_TEST else BENCH
print(f"benchmark: {len(BENCH)} item | evaluated: {len(ITEMS)} | data: {DATA_SRC}")
print(pd.read_csv(f"{BENCH_DIR}/oracle_summary.csv").iloc[:, :6].to_string(index=False))

In [ ]:
# ===================== 5. B3 — STATIC RULES (no LLM) =====================
def frame_name(code):
    m = re.search(r"^\s*(\w+) = df\.copy\(\)", code, re.M)
    return m.group(1) if m else "df"

def in_loop(lines, i):
    """Is line i (1-based) inside a for-loop?"""
    cur = lines[i - 1]
    ind = len(cur) - len(cur.lstrip())
    for j in range(i - 2, -1, -1):
        l = lines[j]
        if not l.strip():
            continue
        if len(l) - len(l.lstrip()) < ind:
            return l.strip().startswith("for ")
    return False

def static_findings(code):
    """Generic rules derived from the taxonomy. Returns [{'type', 'line'}]."""
    D = re.escape(frame_name(code))
    lines = code.splitlines()
    out = []
    for i, ln in enumerate(lines, 1):
        s = ln.strip()
        if not s or s.startswith("#"):
            continue
        lhs, rhs = (s.split("=", 1) + [""])[:2] if "=" in s else ("", s)
        is_target = re.search(r"\[['\"]y['\"]\]\s*$", lhs.strip()) is not None
        t = None
        if ("fillna" in s or "interpolate" in s or "bfill" in s) and \
           ("interpolate" in s or "bfill" in s or "center=True" in s):
            t = "L5"
        elif not is_target and (re.search(r"shift\(\s*-", s) or "center=True" in s or "[::-1].rolling" in s):
            t = "L3"
        elif not is_target and re.search(rf"{D}\[['\"]y['\"]\]", rhs) and "shift(" not in s \
                and not re.match(r"X\s*,\s*y\b", lhs.strip()):
            t = "L4"
        elif ("train_test_split(" in s and "shuffle=False" not in s) or re.search(r"\.sample\(\s*frac", s):
            t = "L2"
        elif re.search(r"Scaler\(\)\.fit(_transform)?\(\s*(X\s*\)|pd\.concat)", s):
            t = "L1"
        elif re.search(r"KFold\(.*shuffle\s*=\s*True", s) or "ShuffleSplit(" in s \
                or re.search(r"cross_val_score\(.*cv\s*=\s*\d", s):
            t = "L6"
        elif "y_te" in s and ("for p in" in s or in_loop(lines, i)):
            t = "L7"
        elif re.search(rf"STL\(\s*{D}\[", s) or re.search(rf"\b{D}\.groupby\(", s):
            t = "L8"
        if t:
            out.append({"type": t, "line": i})
    return out

def b3_predict(item):
    f = static_findings(item["code"])
    return dict(pred_leak=int(bool(f)), pred_type=f[0]["type"] if f else None,
                pred_line=f[0]["line"] if f else None, parse_ok=True, latency=0.0)

B3 = [dict(id=it["id"], model="static", method="B3", **b3_predict(it)) for it in ITEMS]
print("B3 done:", sum(r["pred_leak"] for r in B3), "alarms")

In [ ]:
# ===================== 6. PROMPTS =====================
TAXONOMY = """L1 Preprocessing leakage: a scaler/normaliser is fitted on data that includes the test period.
L2 Split leakage: a time series is split randomly/shuffled instead of chronologically.
L3 Look-ahead feature: a predictor uses values from later time steps (negative shift, centred or forward-looking windows).
L4 Target leakage: a predictor is computed from the target at the same, unlagged step.
L5 Imputation leakage: missing values are filled using future observations (backward fill, two-sided interpolation, centred windows).
L6 Cross-validation leakage: performance is reported with shuffled or non-temporal cross-validation.
L7 Model-selection leakage: hyper-parameters or models are chosen by their score on the test set.
L8 Decomposition/climatology leakage: seasonal decomposition, climatology or normalisation statistics are computed on the full series including the test period."""

NOT_LEAK = ("Correct patterns are NOT leakage: lags with positive shift, trailing windows, the target "
            "definition y = Q.shift(-H), chronological splits (shuffle=False or date cut), TimeSeriesSplit, "
            "statistics fitted on the training part only.")

SYS = ("You are a senior machine-learning engineer reviewing Python code of a time-series forecasting "
       "pipeline for temporal data leakage. The forecast target is y(t) = Q(t+H). "
       "Reply with exactly one JSON object and nothing else.")

def numbered(code):
    return "\n".join(f"{i:>3}| {l}" for i, l in enumerate(code.splitlines(), 1))

def make_prompt(method, item):
    code = f"```python\n{numbered(item['code'])}\n```"
    if method == "B0":
        return (f"Does the following code contain data leakage that makes the reported score optimistic?\n\n{code}\n\n"
                'Return JSON: {"leakage": true|false, "line": <line number of the leaking statement or null>}')
    schema = ('{"leakage": true|false, "type": "L1".."L8" or null, "line": <line number or null>, '
              '"fix": "<corrected code for that line>" or null}')
    if method == "B1":
        return (f"Leakage taxonomy:\n{TAXONOMY}\n\n{NOT_LEAK}\n\nReview this code:\n\n{code}\n\n"
                f"If there are several leaks, report the most severe one. Return JSON: {schema}")
    schema_cot = ('{"analysis": "<max 5 sentences>", "leakage": true|false, "type": "L1".."L8" or null, '
                  '"line": <line number or null>, "fix": "<corrected code for that line>" or null}')
    cot = ("First, in the field \"analysis\", trace how information from later time steps or from the test "
           "period could flow into features, fitted statistics, model selection or the reported score. Then decide.")
    if method == "B2":
        return (f"Leakage taxonomy:\n{TAXONOMY}\n\n{NOT_LEAK}\n\nReview this code:\n\n{code}\n\n{cot}\n"
                f"Return JSON: {schema_cot}")
    if method == "B4":
        f = static_findings(item["code"])
        if f:
            rep = "\n".join(f"- line {x['line']}: possible {x['type']}" for x in f)
            rep = ("A static analyzer produced these findings. It can be wrong: verify each finding against "
                   f"the code, and also look for leakage it cannot detect.\n{rep}")
        else:
            rep = "A static analyzer reported no findings. It may miss leakage, so review the code yourself."
        return (f"Leakage taxonomy:\n{TAXONOMY}\n\n{NOT_LEAK}\n\nReview this code:\n\n{code}\n\n{rep}\n\n{cot}\n"
                f"Return JSON: {schema_cot}")
    raise ValueError(method)

In [ ]:
# ===================== 7. PARSING MODEL OUTPUT =====================
def clean_fix(fix):
    """Strip ``` fences, a leading 'python' and '27| ' line numbers from an LLM patch."""
    if not isinstance(fix, str):
        return None
    f = re.sub(r"```(?:python|py)?", "", fix).strip("\n")
    f = "\n".join(re.sub(r"^\s*\d+\s*\|\s?", "", l) for l in f.split("\n"))
    f = re.sub(r"^\s*python\s*\n", "", f)
    return f if f.strip() else None

def parse(txt):
    obj, ok = None, False
    txt = re.sub(r"```(?:json)?", "", txt)
    m = re.search(r"\{.*\}", txt, re.S)
    if m:
        for cand in (m.group(0), re.sub(r"\bTrue\b", "true", re.sub(r"\bFalse\b", "false",
                                     re.sub(r"\bNone\b", "null", m.group(0))))):
            try:
                obj, ok = json.loads(cand), True
                break
            except Exception:
                pass
    if obj is None:  # fallback: extract fields with regular expressions
        obj = {}
        lk = re.search(r'"?leakage"?\s*[:=]\s*(true|false)', txt, re.I)
        obj["leakage"] = (lk.group(1).lower() == "true") if lk else ("leak" in txt.lower() and "no leak" not in txt.lower())
        ty = re.search(r"\bL([1-8])\b", txt)
        obj["type"] = f"L{ty.group(1)}" if ty else None
        li = re.search(r'"?line"?\s*[:=]\s*(\d+)', txt)
        obj["line"] = int(li.group(1)) if li else None
    leak = obj.get("leakage")
    if isinstance(leak, str):
        leak = leak.strip().lower() in ("true", "yes", "1")
    typ = obj.get("type")
    typ = (re.search(r"L[1-8]", str(typ).upper()).group(0)
           if typ and re.search(r"L[1-8]", str(typ).upper()) else None)
    try:
        line = int(obj.get("line")) if obj.get("line") is not None else None
    except (TypeError, ValueError):
        line = None
    fix = clean_fix(obj.get("fix"))
    return dict(pred_leak=int(bool(leak)), pred_type=typ if leak else None,
                pred_line=line if leak else None, fix=fix if (leak and isinstance(fix, str)) else None,
                parse_ok=ok)

def vote(parsed):
    """Self-consistency: majority vote + confidence."""
    if len(parsed) == 1:
        return dict(parsed[0], conf=1.0)
    leaks = [p["pred_leak"] for p in parsed]
    maj = int(sum(leaks) * 2 >= len(leaks))
    pool = [p for p in parsed if p["pred_leak"] == maj]
    best = max(pool, key=lambda p: sum((q["pred_type"], q["pred_line"]) == (p["pred_type"], p["pred_line"]) for q in pool))
    return dict(best, conf=len(pool) / len(parsed), parse_ok=all(p["parse_ok"] for p in parsed))

In [ ]:
# ===================== 8. MODEL AND GENERATION =====================
def load_model(name):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    mem = sum(torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())) / 1e9
    pb = re.search(r"(\d+(?:\.\d+)?)[bB]", name.split("/")[-1])
    need = float(pb.group(1)) * 2.3 if pb else 99
    kw = dict(device_map="auto", torch_dtype=dtype)
    if need > mem * 0.8:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=dtype)
        print(f"{name}: 4-bit (GPU {mem:.0f} GB)")
    else:
        print(f"{name}: {dtype} (GPU {mem:.0f} GB)")
    model = AutoModelForCausalLM.from_pretrained(name, **kw).eval()
    return tok, model

def generate(tok, model, prompts, sys_prompt=None, max_new=None, k=None):
    import torch
    sys_prompt = sys_prompt or SYS
    k = k or SC_K
    texts = [tok.apply_chat_template([{"role": "system", "content": sys_prompt}, {"role": "user", "content": p}],
                                     tokenize=False, add_generation_prompt=True) for p in prompts]
    enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
    kw = dict(max_new_tokens=max_new or MAX_NEW_TOKENS, pad_token_id=tok.pad_token_id, num_return_sequences=k)
    kw.update(dict(do_sample=True, temperature=0.7, top_p=0.95) if k > 1 else dict(do_sample=False))
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    dec = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [dec[i * k:(i + 1) * k] for i in range(len(prompts))]

def fake_generate(prompts, items):
    """DRY_RUN: returns JSON from the static rules with added noise."""
    outs = []
    for it in items:
        f = static_findings(it["code"])
        leak = bool(f) if random.random() > 0.15 else not f
        if leak and f:
            o = {"leakage": True, "type": f[0]["type"], "line": f[0]["line"],
                 "fix": it["code"].splitlines()[f[0]["line"] - 1].strip()}
        else:
            o = {"leakage": leak, "type": None, "line": None, "fix": None}
        outs.append([json.dumps(o)] * SC_K)
    return outs

def short(name):
    return name.split("/")[-1].replace(".", "_")

def run_all():
    for name in MODELS:
        tok = model = None
        todo_any = False
        for meth in METHODS:
            path = f"{RES_DIR}/raw_{short(name)}_{meth}.jsonl"
            done = {r["id"] for r in read_jsonl(path)} if os.path.exists(path) else set()
            todo = [it for it in ITEMS if it["id"] not in done]
            if not todo:
                print(f"{short(name)} {meth}: done ({len(done)})")
                continue
            if model is None and not DRY_RUN:
                tok, model = load_model(name)
            todo_any = True
            t_start = time.time()
            for b in range(0, len(todo), BATCH_SIZE):
                batch = todo[b:b + BATCH_SIZE]
                prompts = [make_prompt(meth, it) for it in batch]
                t0 = time.time()
                outs = fake_generate(prompts, batch) if DRY_RUN else generate(tok, model, prompts)
                lat = (time.time() - t0) / len(batch)
                with open(path, "a", encoding="utf-8") as f:
                    for it, raw in zip(batch, outs):
                        res = vote([parse(r) for r in raw])
                        f.write(json.dumps(dict(id=it["id"], model=short(name), method=meth,
                                                latency=round(lat, 3), raw=raw, **res),
                                           ensure_ascii=False) + "\n")
                n = min(b + BATCH_SIZE, len(todo))
                el = time.time() - t_start
                print(f"\r{short(name)} {meth}: {n}/{len(todo)}  ~{el / n * (len(todo) - n) / 60:.1f} min left",
                      end="")
            print()
        if model is not None:
            import torch
            del model, tok
            gc.collect()
            torch.cuda.empty_cache()

run_all()

In [ ]:
# ===================== 9. METRICS (RQ1–RQ3) =====================
from statsmodels.stats.contingency_tables import mcnemar

meta = pd.DataFrame([{k: it[k] for k in ("id", "label", "leak_type", "hard_negative", "leak_lines")}
                     for it in ITEMS])
rows = list(B3)
for name in MODELS:
    for meth in METHODS:
        p = f"{RES_DIR}/raw_{short(name)}_{meth}.jsonl"
        if os.path.exists(p):
            for r in read_jsonl(p):
                if r["id"] in set(meta.id):
                    r.update(vote([parse(x) for x in r["raw"]]))   # re-parse stored raw outputs
                    rows.append(r)
R = pd.DataFrame(rows).merge(meta, on="id")

# --- Diagnostics: parse failures and share of positive verdicts
diag = R[R.model != "static"].groupby(["model", "method"]).agg(
    parse_fail=("parse_ok", lambda x: round(1 - x.mean(), 3)),
    says_leak=("pred_leak", "mean")).round(3)
print("=== Diagnostics ===")
print(diag.to_string())
for (mdl, meth), g in R[R.model != "static"].groupby(["model", "method"]):
    if g.pred_leak.mean() in (0.0, 1.0) or (1 - g.parse_ok.mean()) > 0.2:
        print(f"\n⚠️ {mdl} {meth}: constant verdict or many parse failures. Sample output:")
        print(str(g.iloc[0]["raw"][0])[:600])
R["correct"] = (R.pred_leak == R.label).astype(int)
R["loc_ok"] = [int(r.pred_line in r.leak_lines) if r.label == 1 else np.nan for r in R.itertuples()]
R["type_ok"] = [int(r.pred_type == r.leak_type) if r.label == 1 else np.nan for r in R.itertuples()]

def f1_of(df):
    tp = ((df.label == 1) & (df.pred_leak == 1)).sum()
    fp = ((df.label == 0) & (df.pred_leak == 1)).sum()
    fn = ((df.label == 1) & (df.pred_leak == 0)).sum()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return p, r, (2 * p * r / (p + r) if p + r else 0.0)

rng_np = np.random.default_rng(SEED)
summary = []
for (mdl, meth), g in R.groupby(["model", "method"]):
    g = g.reset_index(drop=True)
    p, r, f1 = f1_of(g)
    boots = [f1_of(g.iloc[rng_np.integers(0, len(g), len(g))])[2] for _ in range(1000)]
    leaky = g[g.label == 1]
    summary.append(dict(
        model=mdl, method=meth, n=len(g),
        precision=round(p, 3), recall=round(r, 3), f1=round(f1, 3),
        f1_ci_low=round(np.percentile(boots, 2.5), 3), f1_ci_high=round(np.percentile(boots, 97.5), 3),
        accuracy=round(g.correct.mean(), 3),
        fpr_clean=round(g[(g.label == 0) & (~g.hard_negative)].pred_leak.mean(), 3),
        fpr_hard=round(g[g.hard_negative].pred_leak.mean(), 3),
        type_acc=(round(leaky.type_ok.mean(), 3) if meth != "B0" else np.nan),
        loc_acc=round(leaky.loc_ok.mean(), 3),
        parse_fail=round(1 - g.parse_ok.mean(), 3),
        latency_s=round(g.latency.mean(), 2)))
S = pd.DataFrame(summary).sort_values(["model", "method"])
S.to_csv(f"{RES_DIR}/main_results.csv", index=False)
print("=== RQ1/RQ3: main results ===")
print(S.to_string(index=False))

# RQ2: recall per leakage type (heatmap)
T = (R[R.label == 1].assign(col=lambda d: d.model + "|" + d.method)
       .pivot_table(index="leak_type", columns="col", values="pred_leak", aggfunc="mean").round(3))
T.to_csv(f"{RES_DIR}/per_type_recall.csv")
print("\n=== RQ2: recall per type ===")
print(T.to_string())

# McNemar: B4 vs. the other reviewers within each model (+B3)
mc = []
for mdl in R.model.unique():
    if mdl == "static":
        continue
    base = R[(R.model == mdl) & (R.method == "B4")].set_index("id").correct
    if base.empty:
        continue
    others = [(mdl, m) for m in METHODS if m != "B4"] + [("static", "B3")]
    for om, meth in others:
        o = R[(R.model == om) & (R.method == meth)].set_index("id").correct.reindex(base.index)
        if o.isna().any():
            continue
        tab = [[int(((base == 1) & (o == 1)).sum()), int(((base == 1) & (o == 0)).sum())],
               [int(((base == 0) & (o == 1)).sum()), int(((base == 0) & (o == 0)).sum())]]
        mc.append(dict(model=mdl, compare=f"B4 vs {meth}", b4_only=tab[0][1], other_only=tab[1][0],
                       p_value=round(mcnemar(tab, exact=True).pvalue, 4)))
MC = pd.DataFrame(mc)
MC.to_csv(f"{RES_DIR}/mcnemar.csv", index=False)
print("\n=== McNemar (exact test) ===")
print(MC.to_string(index=False))

In [ ]:
# ===================== 10. RQ4 — REPAIR (oracle-localised) =====================
# The model is given the leaking line and the leakage type and returns only the replacement code.
# A patch is executed and checked: (1) syntax, (2) it runs, (3) the static rule no longer fires,
# (4) NSE does not exceed the clean twin. End-to-end = detected by B4 + correct line + accepted patch.
SYS_R = ("You repair temporal data leakage in Python time-series forecasting code. "
         "Reply with Python code only, without explanations.")
REPAIR_MAX_NEW = 160

def repair_prompt(item):
    n = item["leak_lines"][0]
    return (f"```python\n{numbered(item['code'])}\n```\n\n"
            f"Line {n} causes temporal data leakage ({G.LEAK_TYPES[item['leak_type']]}).\n"
            f"Write the corrected code that replaces line {n}.\n"
            "Rules: output only valid Python code (one or a few lines) inside a ```python block; keep the existing "
            "variable names; you may use variables defined earlier in the code; use only information from the past "
            "or from the training period; no line numbers, no comments, no explanations. "
            "If the line should simply be removed, output: pass")

def extract_code(txt):
    m = re.search(r"```(?:python|py)?\s*\n(.*?)```", txt, re.S)
    return clean_fix(m.group(1) if m else txt)

class _Timeout(Exception):
    pass

def _alarm(signum, frame):
    raise _Timeout()

KNOWN_IMPORTS = {
    "TimeSeriesSplit": "from sklearn.model_selection import TimeSeriesSplit",
    "train_test_split": "from sklearn.model_selection import train_test_split",
    "KFold": "from sklearn.model_selection import KFold",
    "cross_val_score": "from sklearn.model_selection import cross_val_score",
    "make_pipeline": "from sklearn.pipeline import make_pipeline",
    "StandardScaler": "from sklearn.preprocessing import StandardScaler",
    "MinMaxScaler": "from sklearn.preprocessing import MinMaxScaler",
    "STL": "from statsmodels.tsa.seasonal import STL",
}

def apply_fix(code, line_no, fix):
    lines = code.splitlines()
    old = lines[line_no - 1]
    ind = old[: len(old) - len(old.lstrip())]
    fix_lines = [l for l in fix.replace("\r", "").split("\n") if l.strip()]
    base = min(len(l) - len(l.lstrip()) for l in fix_lines)
    lines[line_no - 1: line_no] = [ind + l[base:] for l in fix_lines]
    for nm, imp in KNOWN_IMPORTS.items():   # add imports for standard classes used by the patch
        if re.search(rf"\b{nm}\b", fix) and not re.search(rf"import .*\b{nm}\b", code):
            lines.insert(0, imp)
    return "\n".join(lines) + "\n"

def check_fix(item, fix):
    res = dict(valid=False, runs=False, success=False, nse_fixed=None, err=None)
    if not fix:
        res["err"] = "empty"
        return res
    try:
        ast.parse(textwrap.dedent(fix))
        code2 = apply_fix(item["code"], item["leak_lines"][0], fix)
        ast.parse(code2)
        res["valid"] = True
    except SyntaxError as e:
        res["err"] = type(e).__name__
        return res
    try:
        signal.signal(signal.SIGALRM, _alarm)
        signal.alarm(60)
        res["nse_fixed"] = G.execute(code2, item["style"], DATA_DF)
        signal.alarm(0)
        res["runs"] = True
    except BaseException as e:  # noqa: BLE001
        signal.alarm(0)
        res["err"] = type(e).__name__
        return res
    still = [f for f in static_findings(code2) if f["type"] == item["leak_type"]]
    res["success"] = (not still) and res["nse_fixed"] <= item["nse_twin"] + G.CFG["delta_thr"]
    if still:
        res["err"] = "still_leaky"
    elif not res["success"]:
        res["err"] = "nse_inflated"
    return res

def fake_repair(item):
    tw = TWINS[item["pair_id"]]["code"].splitlines()
    n = item["leak_lines"][0]
    return [f"```python\n{tw[n - 1].strip()}\n```" if len(tw) == len(item["code"].splitlines()) else "pass"]

LEAKY = [it for it in ITEMS if it["label"] == 1]
for name in MODELS:
    path = f"{RES_DIR}/raw_{short(name)}_R.jsonl"
    done = {r["id"] for r in read_jsonl(path)} if os.path.exists(path) else set()
    todo = [it for it in LEAKY if it["id"] not in done]
    if not todo:
        print(f"{short(name)} repair: done ({len(done)})")
        continue
    tok = model = None
    if not DRY_RUN:
        tok, model = load_model(name)
    for b in range(0, len(todo), BATCH_SIZE):
        batch = todo[b:b + BATCH_SIZE]
        outs = ([fake_repair(it) for it in batch] if DRY_RUN else
                generate(tok, model, [repair_prompt(it) for it in batch], SYS_R, REPAIR_MAX_NEW, 1))
        with open(path, "a", encoding="utf-8") as f:
            for it, raw in zip(batch, outs):
                f.write(json.dumps(dict(id=it["id"], model=short(name), raw=raw[0]), ensure_ascii=False) + "\n")
        print(f"\r{short(name)} repair: {min(b + BATCH_SIZE, len(todo))}/{len(todo)}", end="")
    print()
    if model is not None:
        import torch
        del model, tok
        gc.collect()
        torch.cuda.empty_cache()

items_by_id = {it["id"]: it for it in ITEMS}
rep = []
for name in MODELS:
    path = f"{RES_DIR}/raw_{short(name)}_R.jsonl"
    if not os.path.exists(path):
        continue
    for r in read_jsonl(path):
        it = items_by_id.get(r["id"])
        if it is None:
            continue
        fix = extract_code(r["raw"])
        rep.append(dict(model=r["model"], id=r["id"], leak_type=it["leak_type"], fix=fix,
                        nse_twin=it["nse_twin"], **check_fix(it, fix)))
REP = pd.DataFrame(rep)
det = R[R.method == "B4"][["model", "id", "pred_leak", "loc_ok"]]
REP = REP.merge(det, on=["model", "id"], how="left")
REP["end_to_end"] = (REP.pred_leak == 1) & (REP.loc_ok == 1) & REP.success
REP.to_csv(f"{RES_DIR}/repair_items.csv", index=False)

RS = REP.groupby("model").agg(n=("id", "size"), valid_syntax=("valid", "mean"), runs=("runs", "mean"),
                              repair_success=("success", "mean"), end_to_end_B4=("end_to_end", "mean")).round(3)
RS.to_csv(f"{RES_DIR}/repair_summary.csv")
RT = REP.pivot_table(index="leak_type", columns="model", values="success", aggfunc="mean").round(3)
RT.to_csv(f"{RES_DIR}/repair_per_type.csv")
print("=== RQ4: repair ===")
print(RS.to_string())
print("\nRepair success per type:")
print(RT.to_string())
print("\nRejection reasons:")
print(REP.err.value_counts(dropna=True).to_string())
for b in REP[REP.err.notna()].head(3).itertuples():
    print(f"  {b.model} {b.id} {b.leak_type} ({b.err}): {str(b.fix)[:160]!r}")

In [ ]:
# ===================== 11. FILES FOR THE MATLAB FIGURES =====================
import shutil
shutil.copy(f"{BENCH_DIR}/oracle_summary.csv", f"{RES_DIR}/oracle_summary.csv")
pd.DataFrame([{k: it[k] for k in ("id", "pair_id", "leak_type", "variant", "delta_nse", "nse_twin")}
              for it in BENCH if it["label"] == 1]).to_csv(f"{RES_DIR}/oracle_items.csv", index=False)
if IN_KAGGLE:   # zip all outputs (downloadable from the Output tab)
    shutil.make_archive("/kaggle/working/tleakbench_results", "zip", DRIVE_DIR)
    print("zip: /kaggle/working/tleakbench_results.zip")
print("Result files:")
for f in sorted(os.listdir(RES_DIR)):
    if f.endswith(".csv"):
        print("  ", f)